In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

sys.path.append(os.path.abspath(".."))
from feature_extraction.features import extract_all_features

: 

In [ ]:
DATA_DIR = "../data/augmented/train"

CLASSES = [
    "Machinery Fault", "Glass Breaking", "Alarm or Siren", "Vehicle Horn",
    "Animal Sound", "Gunshot", "Panic Scream", "Aggression",
    "Person Asking for Help", "Background Noise"
]

X_data = []
y_data = []

print("Extracting features... this might take a while depending on data size!")
for label in CLASSES:
    class_folder = os.path.join(DATA_DIR, label)
    
    if not os.path.exists(class_folder):
        continue
        
    for file_name in os.listdir(class_folder):
        if file_name.endswith(".wav"):
            file_path = os.path.join(class_folder, file_name)
            
            try:
                features_dict = extract_all_features(file_path)
                X_data.append(features_dict)
                y_data.append(label)
            except Exception as e:
                pass

df = pd.DataFrame(X_data)
X = df.values
y = np.array(y_data)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [500, 1000]
}

In [ ]:
lr_model = LogisticRegression(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(lr_model, param_grid, cv=5, n_jobs=-1, scoring='accuracy')

print("Training the model...")
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("\n--- RESULTS ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

print("Generating confusion matrix plot...")

cm = confusion_matrix(y_test, y_pred, labels=CLASSES)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASSES, yticklabels=CLASSES)

plt.title('SonicSentinel AI - Logistic Regression Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Sound Class', fontsize=12)
plt.ylabel('Actual Sound Class', fontsize=12)

plt.xticks(rotation=45, ha='right') 
plt.tight_layout()

os.makedirs("../reports", exist_ok=True)
plt.savefig("../reports/lr_confusion_matrix.png", dpi=300)

plt.show() 

os.makedirs("../python_models", exist_ok=True)
joblib.dump(best_model, "../python_models/logistic_regression_best.pkl")
joblib.dump(scaler, "../python_models/feature_scaler.pkl")